In [ ]:
-- APB - BRK Data for Panels 3 and 12
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT TOP (200)
      [PnlNo]
      ,[DeviceNo]
      ,[EDate]
      ,[Badge]
      ,[Class]
      ,[Description]
      ,[Name]
      
  FROM [CardAccessArchiveEventsUS].[dbo].[Event]
  WHERE [Class] = 'BADGE Violate APB'
  and EDate BETWEEN '2022-08-01' and '2022-10-25 20:49:20'
  --and PnlNo IN('3')
  --and DeviceNo IN ('3','4')
  and PnlNo IN('12')
  and DeviceNo IN ('1','2')
  --ORDER BY EDate DESC

In [ ]:
-- APB - BRK Event (Archive) and Prev Panel Info - v1
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
select v3.Badge as Badge,v3.[Description] as Agent, vtable3.violationdatetime as APBTime, vtable3.PnlNo as APBPanel, vtable3.DeviceNo as APBDevice, v3.Edate as PriorEvent, v3.PnlNo as PriorPanel, v3.DeviceNo as PriorDevice  
from [CardAccessArchiveEventsUS].[dbo].[Event] v3
inner join
(
select
v2.Badge, Max(v2.EDate) as priordatetime, (vtable.EDate) as violationdatetime, vtable.PnlNo, vtable.DeviceNo
FROM [CardAccessArchiveEventsUS].[dbo].[Event] v2
inner join  
(
SELECT TOP (2000)
 [EDate]
,Badge
,PnlNo
,DeviceNo
 FROM [CardAccessArchiveEventsUS].[dbo].[Event]
 WHERE [Class] = 'BADGE Violate APB'
 --and PnlNo IN('12')
 --and DeviceNo IN ('1','2')
  order by EDate desc
) vtable on vtable.Badge = v2.Badge
  where vtable.EDate > v2.EDate
  group by v2.Badge, vtable.EDate, vtable.DeviceNo, vtable.PnlNo
 ) vtable3 on v3.EDate = vtable3.priordatetime and vtable3.Badge = v3.Badge
where v3.Class != 'BADGE Violate APB'
-- The next line can be used to filter down to a spicific user or badge.
--and v3.[Description] LIKE '%Woodard%'
--and v3.Badge like '17960'
order by APBTime desc


In [ ]:
-- APB - BRK Event (Live) and Prev Panel Info - v1
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
select v3.Badge as Badge,v3.[Description] as Agent, vtable3.violationdatetime as APBTime, vtable3.PnlNo as APBPanel, vtable3.DeviceNo as APBDevice, v3.Edate as PriorEvent, v3.PnlNo as PriorPanel, v3.DeviceNo as PriorDevice  
from [CardAccessLiveEventsUS].[dbo].[Event] v3
inner join
(
select
v2.Badge, Max(v2.EDate) as priordatetime, (vtable.EDate) as violationdatetime, vtable.PnlNo, vtable.DeviceNo
FROM [CardAccessLiveEventsUS].[dbo].[Event] v2
inner join  
(
SELECT TOP (2000)
 [EDate]
,Badge
,PnlNo
,DeviceNo
 FROM [CardAccessLiveEventsUS].[dbo].[Event]
 WHERE [Class] = 'BADGE Violate APB'
 --and PnlNo IN('12')
 --and DeviceNo IN ('1','2')
  order by EDate desc
) vtable on vtable.Badge = v2.Badge
  where vtable.EDate > v2.EDate
  group by v2.Badge, vtable.EDate, vtable.DeviceNo, vtable.PnlNo
 ) vtable3 on v3.EDate = vtable3.priordatetime and vtable3.Badge = v3.Badge
where v3.Class != 'BADGE Violate APB'
-- The next line can be used to filter down to a spicific user or badge.
--and v3.[Description] LIKE '%Woodard%'
and v3.Badge like '15967'
order by APBTime desc


In [ ]:
-- APB - Count of BRK Events from Date Range
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT (
    SELECT COUNT(*) 
    FROM [CardAccessArchiveEventsUS].[dbo].[Event]
    WHERE [Class] = 'BADGE Violate APB'
    and EDate BETWEEN '2022-10-17 07:02:02' and '2022-10-19 14:40'
) as "Total APB"

In [ ]:
-- APB - List of BRK Events in Date Range
-- Run against the SQL Database
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT TOP (2000) 
       [PnlNo]
      ,[DeviceNo]
      ,[EDate]
      ,[Badge]
      ,[Class]
      ,[Description]
      ,[Name]
      
  FROM [CardAccessArchiveEventsUS].[dbo].[Event]
  WHERE [Class] = 'BADGE Violate APB'
  and EDate BETWEEN '2022-11-14 00:00:00.000' and '2022-11-17 00:00:00.000';  

In [ ]:
-- Badge - Employees APB Exempt
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT
      p.FrstName
      ,p.[LastName]
      ,b.[Badge]
      ,b.[Enabled]
      ,b.[Embossed]
      ,b.[MApbExempt] as 'APB Exempt'
      ,b.[SiteNo]
      ,b.[Features]
      ,o.[OperLoginName] AS 'Last Updated by'
      ,b.[LastUpdated]      
  FROM [CardAccessLiveConfigurationUS].[dbo].[Badge] AS b
  LEFT JOIN [CardAccessLiveConfigurationUS].[dbo].[Person] AS p ON b.PersonID = p.PersonID 
  LEFT JOIN [dbo].[Operators] AS o ON b.[LastOperator] = o.[operatorID]
  WHERE b.[MApbExempt] = 1 and b.[Enabled] = 1 and b.[Embossed] > 0


In [ ]:
-- COM Ports - BRK
-- This needs to be run from the SQL server and CardAccessLiveConfiguration database
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT 
--List of Panel Fields
    c.ComPort AS "Com Port #",
    CASE
        WHEN c.Enabled = 0 THEN 'Disabled'
        WHEN c.Enabled = 1 THEN 'Active'
        ELSE 'N/A'
    END AS "COM Active",
    c.IPAddress AS "IP Address",
    c.MACAddress AS "MAC Address",
    p.PanelName AS "Panel Name",
    c.LastUpdated as "Last Updated"    
FROM COM AS c
LEFT JOIN Panel AS p ON c.ComPort = p.ComPort
ORDER BY c.ComPort

In [ ]:
-- Config - BRK COM Ports and related Panels
-- This needs to be run from the SQL server and CardAccessLiveConfiguration database
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT 
--List of Panel Fields
    c.ComPort AS "Com Port #",
    CASE
        WHEN c.Enabled = 0 THEN 'Disabled'
        WHEN c.Enabled = 1 THEN 'Active'
        ELSE 'N/A'
    END AS "COM Active",
    p.PnlNo AS "Panel #",
    c.IPAddress AS "IP Address",
    c.MACAddress AS "MAC Address",
    p.Enabled AS "Panel Active ",
    p.PanelName AS "Panel Name",
    p.Description AS "Panel Description",
    p.LastUpdated AS "Panel Last Updated",
    p.UTCOffset AS "Panel UTCOffset"
    --c.MACAddress AS "COM MAC",
/*List of Reader Fields
    r.Enabled AS "Reader Active",
    r.PnlRef AS "Reader Panel #",
    r.RdrNo AS "Reader #",
    r.ReaderName AS "Reader Name",
    r.LastUpdated AS "Reader Last Updated",
    r.UTCOffset AS "Reader UTCOffset"
    */
FROM COM AS c
LEFT JOIN Panel AS p ON c.ComPort = p.ComPort
ORDER BY c.ComPort

In [ ]:
-- Config - BRK COM Ports for PowerShell
-- Run from the LiveConfig Database
SELECT 
    CONCAT(
        CASE WHEN LEN(c.ComPort) = 1 THEN '0' + CAST(c.ComPort AS VARCHAR(2)) ELSE CAST(c.ComPort AS VARCHAR(2)) END,
        '-',        
        CASE WHEN LEN(p.PnlNo) = 1 THEN '0' + CAST(p.PnlNo AS VARCHAR(2)) ELSE CAST(p.PnlNo AS VARCHAR(2)) END,
        ', ',
        c.IPAddress,
        ', ',
        p.PanelName
    ) AS "Combined Column"
FROM COM AS c
LEFT JOIN Panel AS p ON c.ComPort = p.ComPort
ORDER BY "Combined Column";



In [ ]:
-- Config - BRK Operators with Role
-- Connect to SQL database and run against Live Config
    SELECT 
      o.OperLoginName
      ,o.OperatorID
      ,r.RoleName
      ,o.RoleID
      ,o.OperFName
      ,o.OperLName
      ,o.Enabled
      ,o.EmailID
      ,o.LastLoggedIn
      ,o.LastLoggedInUncName
      ,o.LogoffTime
      ,o.LastUpdated
      ,o.IsGlobalAdministrator
      ,o.CanChangeThreatLevel
      ,o.UseLegacyDisplay
      ,o.ManualPrivileges
      ,o.DisableLogoff
      ,o.EventCount
      ,o.LastOperator
      ,o.LastWorkStation
    FROM Operators AS o
    LEFT JOIN Roles AS r ON o.RoleID = r.RoleID
    ORDER by o.OperLoginName;

-- Get a list of Readers and list who last updated it
    SELECT
        r.RdrNo,
        r.ReaderName,
        r.LastUpdated,
        o.OperLoginName,
        r.LastOperator
    FROM Reader r
    Join Operators AS o ON r.LastOperator = o.OperatorID
    ORDER BY r.LastUpdated DESC;

--Show 5 Readers with columns of data
    SELECT TOP 5
        *
    FROM Reader;

In [ ]:
-- Config - BRK Panels and related COM Ports
-- This needs to be run from the SQL server and CardAccessLiveConfiguration database
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT 
--List of Panel Fields
    p.PnlNo AS "Panel #",
    p.ComPort AS "Com Port #",
    CASE
        WHEN c.Enabled = 0 THEN 'Disabled'
        WHEN c.Enabled = 1 THEN 'Active'
        ELSE 'N/A'
    END AS "COM Active",
    c.IPAddress AS "IP Address",
    c.MACAddress AS "MAC Address",
    p.Enabled AS "Panel Active ",
    p.PanelName AS "Panel Name",
    p.Description AS "Panel Description",
    p.LastUpdated AS "Panel Last Updated",
    p.UTCOffset AS "Panel UTCOffset"
    --c.MACAddress AS "COM MAC",
/*List of Reader Fields
    r.Enabled AS "Reader Active",
    r.PnlRef AS "Reader Panel #",
    r.RdrNo AS "Reader #",
    r.ReaderName AS "Reader Name",
    r.LastUpdated AS "Reader Last Updated",
    r.UTCOffset AS "Reader UTCOffset"
    */
FROM Panel AS p
LEFT JOIN COM AS c ON p.ComPort = c.ComPort
ORDER BY p.PnlNo;


In [ ]:
-- Config - BRK Readers and related Panels
-- This needs to be run from the SQL server and CardAccessLiveConfiguration database
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT 
--List of Fields
    p.[PnlNo] AS "Panel #",
    CASE
        WHEN p.[Enabled] = 0 THEN 'Disabled'
        WHEN p.[Enabled] = 1 THEN 'Active'
        ELSE 'N/A'
    END AS "Panel Active",
    r.[RdrNo] AS "Reader #",
    CASE
        WHEN r.[Enabled] = 0 THEN 'Disabled'
        WHEN r.[Enabled] = 1 THEN 'Active'
        ELSE 'N/A'
    END AS "Reader Active",
    --c.IPAddress AS "IP Address",
    --c.MACAddress AS "MAC Address",
    r.[ReaderName] AS "Reader Name",
    r.[LastUpdated] AS "Reader Last Updated"
FROM [CardAccessLiveConfigurationUS].[dbo].[Reader] AS r
LEFT JOIN [CardAccessLiveConfigurationUS].[dbo].[Panel] AS p ON r.[PnlRef] = p.[PnlNo]
ORDER BY p.[PnlNo],r.[RdrNo];


In [ ]:
-- General - BRK Queries
-- Count of US ArchiveEvents_2 per Reader
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    SELECT TOP 10
        e.PnlNo AS 'Panel Number',
        e.DeviceNo AS 'Reader Number',
        e.name AS 'Event Name',
        COUNT(*) AS 'Event Count'
    FROM [CardAccessArchiveEventsUS_2].[dbo].[Event] e
    --WHERE FORMAT(e.edate, 'YYYY-MM') = '2023-09'
    GROUP BY e.PnlNo, e.DeviceNo, e.name
    ORDER BY 'Panel Number','Reader Number';

--Top 10 US Archive_2 events with all columns visible
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    Select Top (10)
    *
    FROM [CardAccessArchiveEventsUS_2].[dbo].[Event];

--All US readers ordered by panel
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    Select
    r.PnlRef AS 'Panel Number',
    r.RdrNo AS 'Reader Number',
    r.ReaderName AS 'Reader Name',
    r.[Description] AS 'Description'
    FROM [CardAccessLiveConfigurationUS].[dbo].Reader r
    ORDER BY r.PnlRef;

--Top 10 US ArchiveEvents_2 Events with Reader Names from ArchiveConfig Joined with e.DeviceNo = r.RdrNo
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    SELECT Top 10
    e.PnlNo AS 'Event Panel Number',
    e.DeviceNo AS 'Event Reader Number',
    r.ReaderName AS 'Reader Name',
    COUNT(*) AS 'Event Count'
    FROM [CardAccessArchiveEventsUS_2].[dbo].[Event] e
    INNER JOIN [CardAccessArchiveConfigurationUS].[dbo].[Reader] r ON (e.DeviceNo = r.RdrNo AND e.PnlNo = r.PnlRef)
    WHERE e.EDate BETWEEN '2022-10-01' AND  '2022-11-01'
    GROUP BY  e.PnlNo, e.DeviceNo, r.ReaderName;

--Top 10 US ArchiveEvents_2 Events with Reader Names from ArchiveConfig Joined with r.RdrNo = e.DeviceNo
    SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
    SELECT top 10
    e.PnlNo AS 'Panel Number',
    e.DeviceNo AS 'Reader Number',
    r.[Description] AS 'Device Name',
    COUNT(e.EDate) AS 'Event Count'
    FROM [CardAccessArchiveEventsUS].[dbo].[Event] e
    JOIN [CardAccessArchiveConfigurationUS].[dbo].[Reader] r ON r.RdrNo = e.DeviceNo
    --WHERE e.edate >= '2023-10-01' AND e.edate < '2023-11-01'
    GROUP BY e.PnlNo, e.DeviceNo, r.Description;



In [ ]:
-- Status - BRK Xacts
-- RUN from the BRK SQL server
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED
SELECT 
    t1.Panel,
    t4.PanelName,
    t1.Type,
    t1.Device,
        CASE
        WHEN t1.Status = 13 THEN 'Active'
        ELSE 'N/A' 
    END 'Status',
    t1.SDate,
    t1.State,
    t1.Version,
    t1.Xact
FROM CardAccessLiveEventsUS.dbo.Status t1 
LEFT JOIN CardAccessArchiveEventsUS.dbo.Status t2 ON t1.Panel = t2.Panel
LEFT JOIN CardAccessArchiveEventsUS_2.dbo.Status t3 ON t1.Panel = t3.Panel
LEFT JOIN CardAccessLiveConfigurationUS.dbo.Panel t4 On t1.Panel = t4.PnlNo 
WHERE t1.Xact > 0 OR t2.Xact > 0 OR t3.Xact > 0
ORDER By Xact DESC
